# 05 · Model Improvement, Hyperparameter Tuning & Threshold

**Goal:** make the notebook 04 model better *without* losing the ability to explain it, and catch **more leavers**.

Three steps:
1. **Better inputs (feature engineering):** give Logistic Regression the patterns we found in the EDA as ready-made columns
2. **Hyperparameter tuning:** search for the best model settings with `GridSearchCV`
3. **Threshold optimisation:** choose the risk cut-off that catches more churners, and show why

To keep it fair we decide everything with **cross-validation** on the training customers only, and keep the test customers for the very end.

> Run `01_data_types.ipynb` first. The new feature code lives in `churn_utils.py` so the app uses exactly the same steps.

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score, fbeta_score,
                             roc_auc_score, confusion_matrix)
import matplotlib.pyplot as plt

from churn_utils import add_features, encode, NUM_COLS

df = pd.read_pickle("telco_clean.pkl")

# every text column and its possible values (saved with the model, so the app knows them too)
categories = {col: list(df[col].cat.categories) for col in df.select_dtypes(include="category").columns}

train, test = train_test_split(df, test_size=0.2, stratify=df["Churn"], random_state=42)
print("Train:", len(train), "| Test:", len(test))

## What is cross-validation?

One train/test split can be lucky or unlucky. **Cross-validation** gives a fairer score:
1. cut the training customers into 5 parts
2. train on 4 parts, test on the 5th
3. repeat 5 times so every part is tested once
4. take the average score

We compare models by **ROC-AUC** (how well they rank leavers above stayers), averaged over the 5 rounds.

`make_pipeline(StandardScaler(), model)` bundles scaling with the model, so in each round the scaler only learns from that round's training part. No leaking.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_auc(model, X, y):
    scores = cross_val_score(model, X, y, cv=folds, scoring="roc_auc")
    return round(scores.mean(), 4), round(scores.std(), 4)

# inputs exactly like notebook 03 (no new features)
def old_inputs(customers):
    X = customers.drop(columns=["Churn", "customerID", "TotalCharges"])
    return pd.get_dummies(X, drop_first=True, dtype=int)

X_train_old = old_inputs(train)
y_train = train["Churn"]
y_test = test["Churn"]

## Step 1 · Feature engineering

Logistic Regression can only *add up* effects. It can't see on its own that "month-to-month **and** fiber" together is
much worse than each alone. So we hand it these patterns as new columns (code in `churn_utils.py`):

| New column | Meaning | Why (from the EDA) |
|---|---|---|
| `num_addons` | how many of the 6 add-ons they have | more add-ons = more loyal |
| `is_new` | tenure of 6 months or less | 53% churn in the first 6 months |
| `fiber_no_support` | fiber with no TechSupport and no OnlineSecurity | the riskiest service setup |
| `m2m_fiber` | month-to-month **and** fiber | 55% churn |
| `m2m_new` | month-to-month **and** first year | new + no commitment |
| `log_tenure` | log of tenure | churn falls fast in early months, then levels off |

In [ ]:
add_features(train)[["tenure", "Contract", "InternetService", "num_addons", "is_new",
                     "fiber_no_support", "m2m_fiber", "m2m_new", "log_tenure"]].head().round(2)

In [ ]:
X_train_new = encode(train, categories)

rows = []
for name, X in [("Current inputs", X_train_old), ("With new features", X_train_new)]:
    model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
    mean, std = cv_auc(model, X, y_train)
    rows.append([name, X.shape[1], mean, std])

feature_table = pd.DataFrame(rows, columns=["Logistic Regression", "input columns",
                                            "cv ROC-AUC (average)", "spread (+/-)"])
feature_table

**Finding:** the new features lift Logistic Regression above the tree models from notebook 03 (about 0.848),
and the model stays just as easy to explain.

## Step 2 · Hyperparameter tuning

A **hyperparameter** is a setting we choose *before* training (the model doesn't learn it from data).
For Logistic Regression the important ones are:

| Setting | What it controls |
|---|---|
| `C` | How strongly we stop the model from over-trusting any single factor. **Small C** = more cautious, simpler model; **large C** = follows the training data more closely (risk of overfitting) |
| `l1_ratio` | The type of caution. **0 (L2)** shrinks all weights a little; **1 (L1)** can switch useless columns off completely (sets their weight to 0) |

`GridSearchCV` tries **every combination** (7 values of C x 2 types = 14 models), scores each one with the same
5-fold cross-validation, and keeps the best. Only training customers are used.

In [ ]:
search_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=5000))

settings_to_try = {
    "logisticregression__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
    "logisticregression__l1_ratio": [0, 1],          # 0 = L2, 1 = L1
}

search = GridSearchCV(search_model, settings_to_try, cv=folds, scoring="roc_auc")
search.fit(X_train_new, y_train)

tuning = pd.DataFrame(search.cv_results_)[["param_logisticregression__C",
                                           "param_logisticregression__l1_ratio",
                                           "mean_test_score", "std_test_score"]]
tuning.columns = ["C", "l1_ratio (0=L2, 1=L1)", "cv ROC-AUC", "spread (+/-)"]
tuning = tuning.round(4).sort_values("cv ROC-AUC", ascending=False).reset_index(drop=True)
tuning

In [ ]:
best_C = search.best_params_["logisticregression__C"]
best_l1_ratio = search.best_params_["logisticregression__l1_ratio"]
print(f"Best settings: C = {best_C}, l1_ratio = {best_l1_ratio}  ->  cv ROC-AUC {search.best_score_:.4f}")

# how much did tuning help compared with the default settings (C = 1, L2)?
default_score = tuning[(tuning["C"] == 1) & (tuning["l1_ratio (0=L2, 1=L1)"] == 0)]["cv ROC-AUC"].iloc[0]
print(f"Default settings (C = 1, L2):            cv ROC-AUC {default_score:.4f}")

In [ ]:
for ratio, label in [(0, "L2"), (1, "L1")]:
    part = tuning[tuning["l1_ratio (0=L2, 1=L1)"] == ratio].sort_values("C")
    plt.plot(part["C"], part["cv ROC-AUC"], marker="o", label=label)
plt.xscale("log")
plt.xlabel("C (log scale): small = more cautious, large = follows the data more")
plt.ylabel("cv ROC-AUC")
plt.title("Tuning results")
plt.legend()
plt.show()

**How to read it:** the curve is almost flat around the best value. The model is **not sensitive** to these settings,
which is good news: it is stable. Tuning gives a small, safe gain, and we now *know* the settings are right instead of guessing.

## Train the final tuned model

Same steps as notebook 04 (median imputation and scaling learned from **train only**), with the new columns and the best settings.

In [ ]:
X_train = encode(train, categories)
X_test = encode(test, categories, columns=list(X_train.columns))

imputer = SimpleImputer(strategy="median").fit(X_train[NUM_COLS])
X_train[NUM_COLS] = imputer.transform(X_train[NUM_COLS])
X_test[NUM_COLS] = imputer.transform(X_test[NUM_COLS])

scaler = StandardScaler().fit(X_train[NUM_COLS])
X_train[NUM_COLS] = scaler.transform(X_train[NUM_COLS])
X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])

new_model = LogisticRegression(C=best_C, l1_ratio=best_l1_ratio, class_weight="balanced",
                               solver="liblinear", max_iter=5000)
new_model.fit(X_train, y_train)
print("Tuned model trained on", X_train.shape[1], "input columns.")
print("Columns switched off by L1 (weight = 0):", int((new_model.coef_[0] == 0).sum()))

## Old vs new on the test customers (never seen)

To compare fairly, we also train the old model here on the same split.

In [ ]:
def scores(true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "accuracy_%": round(accuracy_score(true, pred) * 100, 1),
        "recall_%": round(recall_score(true, pred) * 100, 1),
        "precision_%": round(precision_score(true, pred) * 100, 1),
        "f1_%": round(f1_score(true, pred) * 100, 1),
        "roc_auc": round(roc_auc_score(true, prob), 3),
    }

old_model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
old_model.fit(X_train_old, y_train)
X_test_old = old_inputs(test).reindex(columns=X_train_old.columns, fill_value=0)

old_prob = old_model.predict_proba(X_test_old)[:, 1]
new_train_prob = new_model.predict_proba(X_train)[:, 1]
new_prob = new_model.predict_proba(X_test)[:, 1]

pd.DataFrame({
    "old model (test)": scores(y_test, old_prob),
    "new model (train)": scores(y_train, new_train_prob),
    "new model (test)": scores(y_test, new_prob),
})

- **new vs old (test):** the improvement on unseen customers
- **new train vs new test:** still close, so the new features did not cause overfitting

## What do the new features add?

Their weights (odds ratio above 1 = raises churn, below 1 = lowers it):

In [ ]:
weights = pd.DataFrame({"feature": X_train.columns, "weight": new_model.coef_[0]})
weights["odds_ratio"] = np.exp(weights["weight"]).round(2)

new_feature_names = ["num_addons", "is_new", "fiber_no_support", "m2m_fiber", "m2m_new", "log_tenure"]
weights[weights["feature"].isin(new_feature_names)].sort_values("weight", ascending=False)

## Step 3 · Threshold optimisation: catching more churners

The model gives each customer a risk score. We say "will leave" when the score passes a **threshold** (cut-off).
So far we used the default, **0.5**.

Our goal is to **catch as many leavers as possible**, because the two mistakes don't cost the same:
- **Missing a leaver** loses about a year of their bill (on average about $74/month x 12 = **$890**)
- **A false alarm** only costs an offer we didn't need to give

So a **lower threshold** (catch more leavers, accept more false alarms) should pay off. But how low?

**Fair method:** we choose the threshold on the **training customers** using cross-validation predictions
(`cross_val_predict`: every training customer is scored by a model that did *not* see them).
The test customers stay untouched until the final check.

We judge every threshold three ways:
1. **Recall:** % of leavers caught
2. **F2 score:** a standard score like F1, but it counts **recall twice as much as precision**. Made for exactly this situation
3. **Money:** net value of the retention campaign (assumptions below)

In [ ]:
# business assumptions (change them to test other scenarios)
SAVE_RATE = 0.25     # we keep 1 in 4 of the leavers we contact
OFFER_COST = 50      # average cost of an offer per flagged customer ($ per year)

# same tuned model, scored with cross-validation on the training customers
cv_model = make_pipeline(StandardScaler(),
                         LogisticRegression(C=best_C, l1_ratio=best_l1_ratio, class_weight="balanced",
                                            solver="liblinear", max_iter=5000))
cv_prob = cross_val_predict(cv_model, X_train_new, y_train, cv=folds, method="predict_proba")[:, 1]

yearly_bill = train["MonthlyCharges"].to_numpy() * 12
left = y_train.to_numpy() == 1

rows = []
for t in np.arange(0.20, 0.71, 0.05):
    t = round(t, 2)
    flagged = cv_prob >= t
    money_kept = SAVE_RATE * yearly_bill[flagged & left].sum()
    offer_cost = OFFER_COST * flagged.sum()
    rows.append({
        "threshold": t,
        "flagged_%": round(flagged.mean() * 100, 1),
        "recall_%": round(recall_score(y_train, flagged) * 100, 1),
        "precision_%": round(precision_score(y_train, flagged) * 100, 1),
        "F2": round(fbeta_score(y_train, flagged, beta=2), 4),
        "net_value_$": round(money_kept - offer_cost),
    })

threshold_table = pd.DataFrame(rows)
threshold_table

In [ ]:
best_f2_threshold = threshold_table.loc[threshold_table["F2"].idxmax(), "threshold"]
best_money_threshold = threshold_table.loc[threshold_table["net_value_$"].idxmax(), "threshold"]

fig, (left_chart, right_chart) = plt.subplots(1, 2, figsize=(13, 4.5))

left_chart.plot(threshold_table["threshold"], threshold_table["recall_%"], marker="o", label="Recall (leavers caught)")
left_chart.plot(threshold_table["threshold"], threshold_table["precision_%"], marker="o", label="Precision (flagged who really leave)")
left_chart.axvline(best_f2_threshold, color="green", linestyle="--", label=f"best F2 = {best_f2_threshold}")
left_chart.axvline(0.5, color="grey", linestyle=":", label="old default 0.5")
left_chart.set_xlabel("Threshold")
left_chart.set_ylabel("%")
left_chart.set_title("Lower threshold = catch more leavers")
left_chart.legend()

right_chart.plot(threshold_table["threshold"], threshold_table["net_value_$"] / 1000, marker="o", color="purple")
right_chart.axvline(best_f2_threshold, color="green", linestyle="--")
right_chart.axvline(0.5, color="grey", linestyle=":")
right_chart.set_xlabel("Threshold")
right_chart.set_ylabel("Net value ($ thousands, training customers)")
right_chart.set_title("Money is almost flat between 0.35 and 0.55")
plt.show()

print("Threshold with the best F2:   ", best_f2_threshold)
print("Threshold with the most money:", best_money_threshold)

In [ ]:
chosen = threshold_table[threshold_table["threshold"] == best_f2_threshold].iloc[0]
default = threshold_table[threshold_table["threshold"] == 0.5].iloc[0]
top_money = threshold_table["net_value_$"].max()

print(f"Moving from 0.5 to {best_f2_threshold}:")
print(f"  recall    {default['recall_%']}% -> {chosen['recall_%']}%   (+{chosen['recall_%'] - default['recall_%']:.1f} points of leavers caught)")
print(f"  precision {default['precision_%']}% -> {chosen['precision_%']}%")
print(f"  money     {chosen['net_value_$'] / top_money * 100:.1f}% of the best possible net value")

**Decision:** we choose the threshold with the **best F2 score**:
- It catches clearly **more leavers** than 0.5 (that was our goal)
- Its money result is **within a few % of the best**, so catching more churners costs us almost nothing
- Going even lower (0.20–0.30) adds only a few more leavers but floods the team with false alarms and the money drops

## Final check on the test customers (never seen)

Old default 0.5 vs our chosen threshold, same tuned model.

In [ ]:
THRESHOLD = float(best_f2_threshold)

pd.DataFrame({
    "threshold 0.5": scores(y_test, new_prob, threshold=0.5),
    f"threshold {THRESHOLD}": scores(y_test, new_prob, threshold=THRESHOLD),
})

In [ ]:
cm = confusion_matrix(y_test, (new_prob >= THRESHOLD).astype(int))
pd.DataFrame(cm, index=["Really stayed", "Really left"],
             columns=["Predicted stay", "Predicted leave"])

## Save the final model (used by the simulation and the app)

In [ ]:
joblib.dump({
    "model": new_model,
    "imputer": imputer,
    "scaler": scaler,
    "columns": list(X_train.columns),
    "categories": categories,
    "threshold": THRESHOLD,
}, "churn_model_v2.joblib")
print("Saved churn_model_v2.joblib")

## Key results

### Step 1 · Feature engineering (5-fold cross-validation)

| Logistic Regression | cv ROC-AUC |
|---|---|
| Original inputs (notebook 04) | 0.845 |
| **+ 6 engineered features** | **0.850** |

This is now above the best tree models from notebook 03 (about 0.848), and the model is still easy to explain.

### Step 2 · Hyperparameter tuning (GridSearchCV, 14 combinations)
- Best: **C = 0.1 with L1**: cv AUC **0.8504**, vs 0.8499 with the default settings.
- The gain is small and the tuning curve is flat. The model is **stable**: it doesn't depend on lucky settings.
- Bonus: L1 switched off **15 columns** that added nothing (weight = 0), so the model is simpler.
  This also removed the confusing overlapping weights (e.g. `num_addons` is now 0, not a misleading "raises churn").

### Step 3 · Threshold: 0.5 → **0.35**

Chosen on the training customers with cross-validation, then checked on the test customers:

| Test customers | Threshold 0.5 | **Threshold 0.35** |
|---|---|---|
| **Recall** (leavers caught) | 79.7% | **89.6%** |
| Precision | 51.9% | 45.0% |
| Accuracy | 75.0% | 68.1% |
| F1 | 62.9% | 59.9% |
| ROC-AUC | 0.848 | 0.848 (same model) |

**Why 0.35:**
- It has the **best F2 score**, the standard metric when catching positives matters more than avoiding false alarms.
- We catch about **9 in 10 leavers** instead of 8 in 10.
- The money result is **96% of the best possible**, because the value curve is nearly flat between 0.35 and 0.55. So catching
  **10 points more churners costs almost nothing**.
- Going lower (0.20–0.30) adds few extra leavers and many more false alarms.

**Expected question: "accuracy went down, isn't that worse?"** No. Accuracy counts every customer the same, but a missed
leaver costs a year of revenue while a false alarm only costs an offer. We deliberately trade a little accuracy for many more caught leavers.

**Honest note:** the money figures rest on assumptions (we keep 1 in 4 contacted leavers, $50 average offer cost).
They are in one cell and easy to change. Better results would need **new data** (support calls, complaints, outages, usage).

The final model and the 0.35 threshold are saved in `churn_model_v2.joblib` and used by notebook 06 and the app.
